# 02 — OT-CFM Evaluation: FID, IS, Runtime

Same evaluation protocol as DDPM — compute FID, IS, and runtime at NFE = 10, 20, 50, 100.

**What this notebook does:**
1. Load trained OT-CFM checkpoint from Google Drive
2. Generate 10k images at each NFE using Midpoint and RK4 solvers
3. Compute FID and IS
4. Measure runtime (sec/image)
5. Save results to Drive

**Run on Colab with GPU.**

In [ ]:
import math
import time
import json
import os
import subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from dataclasses import dataclass
import gc

try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
except ModuleNotFoundError:
    pass

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

try:
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore
except ImportError:
    subprocess.check_call(['pip', 'install', '-q', 'torchmetrics[image]'])
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.inception import InceptionScore

In [ ]:
DRIVE_DIR       = '/content/drive/MyDrive/flow-matching'
CFM_RUN_DIR     = f'{DRIVE_DIR}/runs/cfm'
METRICS_DIR     = f'{CFM_RUN_DIR}/metrics'
EVAL_IMAGES_DIR = f'{CFM_RUN_DIR}/samples/eval'

os.makedirs(METRICS_DIR,     exist_ok=True)
os.makedirs(EVAL_IMAGES_DIR, exist_ok=True)

NFE_POINTS      = [10, 20, 50, 100]
NUM_EVAL_IMAGES = 10000
EVAL_BATCH_SIZE = 256
EVAL_SEED       = 42

---
## Step 1: Model + Solver

In [ ]:
@dataclass
class CFMConfig:
    image_size: int = 32
    channels: int = 3
    model_channels: int = 128
    num_heads: int = 4
    dropout: float = 0.1
    num_groups: int = 32
    sigma_min: float = 0.0
    batch_size: int = 128
    lr: float = 2e-4
    num_epochs: int = 500
    grad_clip: float = 1.0
    ema_decay: float = 0.9999
    clip_denoised: bool = True

cfg = CFMConfig()


class TimeEmbedding(nn.Module):
    def __init__(self, base_dim):
        super().__init__()
        self.base_dim = base_dim
        self.mlp = nn.Sequential(nn.Linear(base_dim, base_dim*4), nn.SiLU(), nn.Linear(base_dim*4, base_dim*4))
    def forward(self, t):
        half  = self.base_dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
        emb   = t[:, None].float() * freqs[None]
        emb   = torch.cat([emb.sin(), emb.cos()], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, num_groups=32, dropout=0.1):
        super().__init__()
        self.norm1    = nn.GroupNorm(num_groups, in_ch)
        self.conv1    = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_emb_dim, out_ch * 2))
        self.norm2    = nn.GroupNorm(num_groups, out_ch)
        self.dropout  = nn.Dropout(dropout)
        self.conv2    = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.act      = nn.SiLU()
        self.shortcut = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(self.act(self.norm1(x)))
        scale, shift = self.time_mlp(t_emb).chunk(2, dim=-1)
        h = h * (scale[:, :, None, None] + 1) + shift[:, :, None, None]
        h = self.dropout(self.conv2(self.act(self.norm2(h))))
        return h + self.shortcut(x)

class SelfAttention(nn.Module):
    def __init__(self, channels, num_heads=4, num_groups=32):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim  = channels // num_heads
        self.norm      = nn.GroupNorm(num_groups, channels)
        self.to_qkv    = nn.Conv2d(channels, channels * 3, 1, bias=False)
        self.to_out    = nn.Conv2d(channels, channels, 1)
    def forward(self, x):
        B, C, H, W = x.shape
        h   = self.norm(x)
        qkv = self.to_qkv(h).reshape(B, 3, self.num_heads, self.head_dim, H * W).permute(1, 0, 2, 4, 3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        attn = F.scaled_dot_product_attention(q, k, v).permute(0, 1, 3, 2).reshape(B, C, H, W)
        return x + self.to_out(attn)

class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, stride=2, padding=1)
    def forward(self, x): return self.conv(x)

class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 3, padding=1)
    def forward(self, x): return self.conv(F.interpolate(x, scale_factor=2, mode='nearest'))

class UNet(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        ch, t_dim, ng, nh, do = cfg.model_channels, cfg.model_channels*4, cfg.num_groups, cfg.num_heads, cfg.dropout
        self.time_emb = TimeEmbedding(ch)
        self.init_conv = nn.Conv2d(3, ch, 3, padding=1)
        self.enc1_a = ResBlock(ch,   ch,   t_dim, ng, do); self.enc1_b = ResBlock(ch,   ch,   t_dim, ng, do); self.down1 = Downsample(ch)
        self.enc2_a = ResBlock(ch,   ch*2, t_dim, ng, do); self.enc2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn2 = SelfAttention(ch*2, nh, ng); self.down2 = Downsample(ch*2)
        self.enc3_a = ResBlock(ch*2, ch*2, t_dim, ng, do); self.enc3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.attn3 = SelfAttention(ch*2, nh, ng); self.down3 = Downsample(ch*2)
        self.mid1 = ResBlock(ch*2, ch*2, t_dim, ng, do); self.mid_attn = SelfAttention(ch*2, nh, ng); self.mid2 = ResBlock(ch*2, ch*2, t_dim, ng, do)
        self.up3 = Upsample(ch*2); self.dec3_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec3_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn3 = SelfAttention(ch*2, nh, ng)
        self.up2 = Upsample(ch*2); self.dec2_a = ResBlock(ch*4, ch*2, t_dim, ng, do); self.dec2_b = ResBlock(ch*2, ch*2, t_dim, ng, do); self.dattn2 = SelfAttention(ch*2, nh, ng)
        self.up1 = Upsample(ch*2); self.dec1_a = ResBlock(ch*3, ch,   t_dim, ng, do); self.dec1_b = ResBlock(ch,   ch,   t_dim, ng, do)
        self.out_norm = nn.GroupNorm(ng, ch); self.out_conv = nn.Conv2d(ch, 3, 1)
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        h = self.init_conv(x)
        h = self.enc1_b(self.enc1_a(h, t_emb), t_emb);  s1 = h
        h = self.attn2(self.enc2_b(self.enc2_a(self.down1(h), t_emb), t_emb)); s2 = h
        h = self.attn3(self.enc3_b(self.enc3_a(self.down2(h), t_emb), t_emb)); s3 = h
        h = self.down3(h); h = self.mid1(h, t_emb); h = self.mid_attn(h); h = self.mid2(h, t_emb)
        h = self.dattn3(self.dec3_b(self.dec3_a(torch.cat([self.up3(h), s3], dim=1), t_emb), t_emb))
        h = self.dattn2(self.dec2_b(self.dec2_a(torch.cat([self.up2(h), s2], dim=1), t_emb), t_emb))
        h = self.dec1_b(self.dec1_a(torch.cat([self.up1(h), s1], dim=1), t_emb), t_emb)
        return self.out_conv(F.silu(self.out_norm(h)))

In [ ]:
try:
    from torchdiffeq import odeint
except ImportError:
    subprocess.check_call(['pip', 'install', '-q', 'torchdiffeq'])
    from torchdiffeq import odeint


@torch.no_grad()
def cfm_sample(model, shape, device, method='midpoint', nfe=50, clip_denoised=True):
    model.eval()

    def ode_fn(t, x):
        t_batch = torch.full((shape[0],), t.item(), device=device)
        return model(x, t_batch)

    x0 = torch.randn(shape, device=device)
    t_span = torch.tensor([0.0, 1.0], device=device)

    nfe_per_step = {'midpoint': 2, 'rk4': 4}[method]
    step_size = nfe_per_step / nfe

    x = odeint(ode_fn, x0, t_span, method=method,
               options={'step_size': step_size})[-1]

    if clip_denoised:
        x = x.clamp(-1, 1)
    return x

---
## Step 2: Load checkpoint

In [ ]:
def load_model(checkpoint_name='last.pt', use_ema=True):
    path = f'{CFM_RUN_DIR}/checkpoints/{checkpoint_name}'
    assert os.path.exists(path), f'Checkpoint not found: {path}'
    ckpt  = torch.load(path, map_location=device)
    model = UNet(cfg).to(device)
    if use_ema and 'ema_shadow' in ckpt:
        for name, param in model.named_parameters():
            param.data.copy_(ckpt['ema_shadow'][name])
        print(f'Loaded EMA weights (epoch {ckpt["epoch"]}, loss={ckpt["loss"]:.4f})')
    else:
        model.load_state_dict(ckpt['model'])
        print(f'Loaded model weights (epoch {ckpt["epoch"]})')
    model.eval()
    return model, ckpt

model, checkpoint = load_model('last.pt')

---
## Step 3: Generate images + compute metrics

In [ ]:
def get_real_images(n, batch_size=256):
    dataset = datasets.CIFAR10(root='./data', train=True, download=True,
                               transform=transforms.ToTensor())
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    imgs = []
    for x, _ in tqdm(loader, desc='Loading real images'):
        imgs.append((x * 255).byte())
        if sum(b.shape[0] for b in imgs) >= n:
            break
    return torch.cat(imgs)[:n]


def generate_images(model, n, batch_size, method='midpoint', nfe=50, seed=42):
    torch.manual_seed(seed)
    all_images = []
    total_time = 0.0
    generated  = 0

    while generated < n:
        bs = min(batch_size, n - generated)
        t_start = time.time()
        imgs = cfm_sample(model, (bs, 3, 32, 32), device, method=method, nfe=nfe)
        total_time += time.time() - t_start
        imgs = ((imgs.cpu() + 1) / 2).clamp(0, 1)
        imgs = (imgs * 255).byte()
        all_images.append(imgs)
        generated += bs
        torch.cuda.empty_cache()

    return torch.cat(all_images)[:n], total_time


def compute_fid_is(real_imgs, fake_imgs):
    bs = 256
    fid_metric = FrechetInceptionDistance(feature=2048).to(device)
    for i in range(0, len(real_imgs), bs):
        fid_metric.update(real_imgs[i:i+bs].to(device), real=True)
    for i in range(0, len(fake_imgs), bs):
        fid_metric.update(fake_imgs[i:i+bs].to(device), real=False)
    fid_score = fid_metric.compute().item()
    del fid_metric

    is_metric = InceptionScore().to(device)
    for i in range(0, len(fake_imgs), bs):
        is_metric.update(fake_imgs[i:i+bs].to(device))
    is_mean, is_std = is_metric.compute()
    is_mean, is_std = is_mean.item(), is_std.item()
    del is_metric

    torch.cuda.empty_cache()
    return fid_score, is_mean, is_std

In [ ]:
print('Loading real CIFAR-10 images...')
real_imgs = get_real_images(NUM_EVAL_IMAGES)
print(f'Real images: {real_imgs.shape}, dtype={real_imgs.dtype}')

all_results = []
checkpoint_name = os.path.basename(f'{CFM_RUN_DIR}/checkpoints/last.pt')

# ---------------------------------------------------------------
# Evaluate both solvers at each NFE point
# ---------------------------------------------------------------
for method in ['midpoint', 'rk4']:
    for nfe in NFE_POINTS:
        # RK4 needs NFE divisible by 4
        if method == 'rk4' and nfe % 4 != 0:
            print(f'\nSkipping {method} NFE={nfe} (not divisible by 4)')
            continue

        print(f'\n{"="*50}')
        print(f'Evaluating {method} at NFE={nfe}')
        print(f'{"="*50}')

        fake_imgs, total_sec = generate_images(
            model, n=NUM_EVAL_IMAGES, batch_size=EVAL_BATCH_SIZE,
            method=method, nfe=nfe, seed=EVAL_SEED
        )

        nfe_dir = f'{EVAL_IMAGES_DIR}/{method}_nfe_{nfe}'
        os.makedirs(nfe_dir, exist_ok=True)
        grid = torchvision.utils.make_grid(fake_imgs[:64].float() / 255, nrow=8, padding=2)
        torchvision.utils.save_image(grid, f'{nfe_dir}/sample_grid.png')

        fid, is_mean, is_std = compute_fid_is(real_imgs, fake_imgs)
        sec_per_image = total_sec / NUM_EVAL_IMAGES

        result = {
            'model': 'ot-cfm', 'checkpoint_name': checkpoint_name,
            'nfe': nfe, 'solver_sampler': method,
            'num_generated': NUM_EVAL_IMAGES,
            'fid': round(fid, 4), 'is_mean': round(is_mean, 4), 'is_std': round(is_std, 4),
            'sec_per_image': round(sec_per_image, 6),
            'total_sampling_sec': round(total_sec, 2),
            'seed': EVAL_SEED, 'notes': f'epoch {checkpoint["epoch"]}',
        }
        all_results.append(result)
        print(f'  FID={fid:.2f}  IS={is_mean:.2f}\u00b1{is_std:.2f}  {sec_per_image*1000:.1f}ms/img')

        del fake_imgs
        torch.cuda.empty_cache()
        gc.collect()

print('\nAll evaluations done!')

---
## Step 4: Save metrics

In [ ]:
import datetime

jsonl_path = f'{METRICS_DIR}/metrics_summary.jsonl'
with open(jsonl_path, 'a') as f:
    for r in all_results:
        r['evaluated_at'] = datetime.datetime.now().isoformat()
        f.write(json.dumps(r) + '\n')

json_path = f'{METRICS_DIR}/metrics_summary.json'
with open(json_path, 'w') as f:
    json.dump(all_results, f, indent=2)

fid_results = [{'nfe': r['nfe'], 'solver_sampler': r['solver_sampler'],
                'fid': r['fid'], 'is_mean': r['is_mean'], 'is_std': r['is_std']} for r in all_results]
with open(f'{METRICS_DIR}/fid_results.json', 'w') as f:
    json.dump(fid_results, f, indent=2)

runtime_results = [{'nfe': r['nfe'], 'solver_sampler': r['solver_sampler'],
                    'sec_per_image': r['sec_per_image'],
                    'total_sampling_sec': r['total_sampling_sec']} for r in all_results]
with open(f'{METRICS_DIR}/runtime_results.json', 'w') as f:
    json.dump(runtime_results, f, indent=2)

print(f'Metrics saved to {METRICS_DIR}')

---
## Step 5: Visualize results

In [ ]:
midpoint_results = [r for r in all_results if r['solver_sampler'] == 'midpoint']
rk4_results      = [r for r in all_results if r['solver_sampler'] == 'rk4']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('CFM Evaluation: Midpoint vs RK4', fontsize=13)

# FID vs NFE
axes[0].plot([r['nfe'] for r in midpoint_results], [r['fid'] for r in midpoint_results],
             'o-', color='steelblue', linewidth=2, markersize=8, label='Midpoint')
axes[0].plot([r['nfe'] for r in rk4_results], [r['fid'] for r in rk4_results],
             's-', color='tomato', linewidth=2, markersize=8, label='RK4')
axes[0].set_xlabel('NFE')
axes[0].set_ylabel('FID \u2193')
axes[0].set_title('FID vs NFE')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Runtime vs NFE
axes[1].plot([r['nfe'] for r in midpoint_results],
             [r['sec_per_image']*1000 for r in midpoint_results],
             'o-', color='steelblue', linewidth=2, markersize=8, label='Midpoint')
axes[1].plot([r['nfe'] for r in rk4_results],
             [r['sec_per_image']*1000 for r in rk4_results],
             's-', color='tomato', linewidth=2, markersize=8, label='RK4')
axes[1].set_xlabel('NFE')
axes[1].set_ylabel('ms / image \u2193')
axes[1].set_title('Runtime vs NFE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{METRICS_DIR}/fid_vs_nfe.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print(f'\n{"Model":<8} {"Solver":<12} {"NFE":<6} {"FID":<8} {"IS":<12} {"ms/img":<10}')
print('-' * 60)
for r in sorted(all_results, key=lambda x: (x['solver_sampler'], x['nfe'])):
    fid_str = f'{r["fid"]:.2f}'
    is_str  = f'{r["is_mean"]:.2f}\u00b1{r["is_std"]:.2f}'
    ms_str  = f'{r["sec_per_image"]*1000:.1f}'
    print(f'{r["model"]:<8} {r["solver_sampler"]:<12} {r["nfe"]:<6} {fid_str:<8} {is_str:<12} {ms_str:<10}')